# Laboratorio: Búsqueda en entornos complejos
- Nicolás Concuá - 23197
- Esteban Carcamo - 23016

## Búsqueda Local 

En este laboratorio se estudiarán algoritmos de búsqueda local y heurística en un entorno de espacio de estados complejo. El problema elegido es el de las 8 reinas, en el cual se deben ubicar 8 reinas sobre un tablero de ajedrez de 8x8 de manera que ninguna reina pueda atacar a otra. 

A diferencia de los métodos de búsqueda sistemática, los algoritmos de búsqueda local no construyen explícitamente un árbol completo, sino que exploran el espacio de estados moviéndose entre configuraciones vecinas. 

## Descripción del problema

El problema de las 8 reinas consiste en colocar 8 reinas sobre un tablero de 8x8 de forma que:
- no compartan la misma fila
- no compartan la misma columna
- no compartan la misma diagonal.

*Representación del estado*

Cada estado se representará como un arreglo de longitud 8:



In [1]:
estado = [0, 4, 7, 5, 2, 6, 1, 3]

significa:
- en la columna 0 hay una reina en la fila 0,
- en la columna 1 hay una reina en la fila 4,
- ...
- en la columna 7 hay una reina en la fila 3.

## Implementación

*En parejas*, deberán implementar: 
- Hill Climbing
- Una variación del Hill Climbing (Stochastic/ First-choice/ Random-restart)
- Beam Search con beam width = _k_

Cada grupo debe realizar experimentos comparativos entre los algoritmos implementados.

### Experimentos requeridos:

Ejecutar cada algoritmo 1000 veces con estados iniciales aleatorios y reportar:

- porcentaje de éxito,
- promedio de largo de episodio (hasta éxito o estancarse)
- promedio de valor heurístico final,
- tiempo promedio de ejecución.

Para Beam Search, repetir los experimentos con distintos valores de _k_ (2,5,10). 


## Funciones útiles

In [2]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


## Hill Climbing

Se genera un estado inicial aleatorio. En cada paso se consideran todos los vecinos (mover una reina a otra fila en su columna). Se elige un vecino con valor heurístico estrictamente menor. Si no hay mejora, el algoritmo se detiene (máximo local o solución).

In [3]:
import random
import time

TAMANO_TABLERO = 8


def generar_estado_aleatorio():
    return [random.randint(0, TAMANO_TABLERO - 1) for i in range(TAMANO_TABLERO)]


def obtener_vecinos(estado):
    vecinos = []
    for col in range(TAMANO_TABLERO):
        fila_actual = estado[col]
        for fila in range(TAMANO_TABLERO):
            if fila != fila_actual:
                vecino = estado[:]
                vecino[col] = fila
                vecinos.append(vecino)
    return vecinos


def hill_climbing(estado_inicial):
    estado = list(estado_inicial)
    pasos = 0
    while True:
        h_actual = heuristica(estado)
        if h_actual == 0:
            return estado, pasos, True
        vecinos = obtener_vecinos(estado)
        mejor_vecino = None
        mejor_h = h_actual
        for v in vecinos:
            h_v = heuristica(v)
            if h_v < mejor_h:
                mejor_h = h_v
                mejor_vecino = v
        if mejor_vecino is None:
            return estado, pasos, False
        estado = mejor_vecino
        pasos += 1

### Experimentos Hill Climbing (1000 ejecuciones)

In [4]:
NUM_EXPERIMENTOS = 1000

exitos = 0
largos = []
heuristicas_finales = []
tiempos = []

for _ in range(NUM_EXPERIMENTOS):
    inicial = generar_estado_aleatorio()
    t0 = time.perf_counter()
    estado_final, largo, exito = hill_climbing(inicial)
    tiempos.append(time.perf_counter() - t0)
    if exito:
        exitos += 1
    largos.append(largo)
    heuristicas_finales.append(heuristica(estado_final))

porcentaje_exito = 100 * exitos / NUM_EXPERIMENTOS
promedio_largo = sum(largos) / NUM_EXPERIMENTOS
promedio_heuristica = sum(heuristicas_finales) / NUM_EXPERIMENTOS
promedio_tiempo = sum(tiempos) / NUM_EXPERIMENTOS

print("Hill Climbing - 1000 ejecuciones con estado inicial aleatorio")
print("-" * 50)
print(f"Porcentaje de éxito:        {porcentaje_exito:.1f}%")
print(f"Promedio largo de episodio: {promedio_largo:.2f}")
print(f"Promedio heurística final:  {promedio_heuristica:.2f}")
print(f"Tiempo promedio (s):        {promedio_tiempo:.6f}")

Hill Climbing - 1000 ejecuciones con estado inicial aleatorio
--------------------------------------------------
Porcentaje de éxito:        13.9%
Promedio largo de episodio: 3.21
Promedio heurística final:  1.27
Tiempo promedio (s):        0.000473


## First-choice Hill Climbing

En cada paso no se evalúan todos los vecinos: se generan en orden aleatorio y se toma el **primer** vecino que mejore la heurística. Si ninguno mejora tras revisar todos, se detiene. Reduce costo por paso cuando hay muchas mejoras posibles y añade aleatoriedad que puede ayudar a escapar de algunos máximos locales.

In [5]:
def first_choice_hill_climbing(estado_inicial):
    estado_actual = list(estado_inicial)
    cantidad_pasos = 0
    while True:
        conflictos_actuales = heuristica(estado_actual)
        if conflictos_actuales == 0:
            return estado_actual, cantidad_pasos, True
        vecinos = obtener_vecinos(estado_actual)
        random.shuffle(vecinos)
        encontro_vecino_mejor = False
        for vecino in vecinos:
            conflictos_vecino = heuristica(vecino)
            if conflictos_vecino < conflictos_actuales:
                estado_actual = vecino
                cantidad_pasos += 1
                encontro_vecino_mejor = True
                break
        if not encontro_vecino_mejor:
            return estado_actual, cantidad_pasos, False

### Experimentos First-choice Hill Climbing (1000 ejecuciones)

In [6]:
NUM_EXPERIMENTOS = 1000

cantidad_exitos = 0
largos_episodio = []
heuristicas_finales_fc = []
tiempos_ejecucion = []

for _ in range(NUM_EXPERIMENTOS):
    estado_inicial = generar_estado_aleatorio()
    inicio = time.perf_counter()
    estado_final, largo_episodio, exito = first_choice_hill_climbing(estado_inicial)
    tiempos_ejecucion.append(time.perf_counter() - inicio)
    if exito:
        cantidad_exitos += 1
    largos_episodio.append(largo_episodio)
    heuristicas_finales_fc.append(heuristica(estado_final))

porcentaje_exito_fc = 100 * cantidad_exitos / NUM_EXPERIMENTOS
promedio_largo_fc = sum(largos_episodio) / NUM_EXPERIMENTOS
promedio_heuristica_fc = sum(heuristicas_finales_fc) / NUM_EXPERIMENTOS
promedio_tiempo_fc = sum(tiempos_ejecucion) / NUM_EXPERIMENTOS

print("First-choice Hill Climbing - 1000 ejecuciones con estado inicial aleatorio")
print("-" * 55)
print(f"Porcentaje de éxito:        {porcentaje_exito_fc:.1f}%")
print(f"Promedio largo de episodio: {promedio_largo_fc:.2f}")
print(f"Promedio heurística final:  {promedio_heuristica_fc:.2f}")
print(f"Tiempo promedio (s):        {promedio_tiempo_fc:.6f}")

First-choice Hill Climbing - 1000 ejecuciones con estado inicial aleatorio
-------------------------------------------------------
Porcentaje de éxito:        14.7%
Promedio largo de episodio: 4.91
Promedio heurística final:  1.28
Tiempo promedio (s):        0.000264


## Beam Search

In [12]:
def beam_search(k, max_pasos=200):

    beam = [generar_estado_aleatorio() for _ in range(k)]

    for pasos in range(max_pasos):

        for estado in beam:
            if heuristica(estado) == 0:
                return estado, pasos, True

        vecinos = []

        for estado in beam:
            vecinos.extend(obtener_vecinos(estado))

        vecinos = [(heuristica(v), v) for v in vecinos]
        vecinos.sort(key=lambda x: x[0])

        beam = [v for (_, v) in vecinos[:k]]

    mejor = min(beam, key=lambda x: heuristica(x))
    return mejor, max_pasos, False

In [13]:
ks = [2,5,10]

for k in ks:

    exitos = 0
    largos = []
    heuristicas_finales = []
    tiempos = []

    for _ in range(1000):

        inicio = time.time()

        estado, pasos, exito = beam_search(k)

        fin = time.time()

        if exito:
            exitos += 1

        largos.append(pasos)
        heuristicas_finales.append(heuristica(estado))
        tiempos.append(fin - inicio)

    print("Beam width:", k)
    print("Porcentaje de éxito:", exitos / 1000)
    print("Promedio largo:", sum(largos)/len(largos))
    print("Promedio heurística:", sum(heuristicas_finales)/len(heuristicas_finales))
    print("Tiempo promedio:", sum(tiempos)/len(tiempos))
    print()

Beam width: 2
Porcentaje de éxito: 0.489
Promedio largo: 104.456
Promedio heurística: 0.573
Tiempo promedio: 0.023876437664031983

Beam width: 5
Porcentaje de éxito: 0.723
Promedio largo: 58.502
Promedio heurística: 0.281
Tiempo promedio: 0.03354090142250061

Beam width: 10
Porcentaje de éxito: 0.876
Promedio largo: 28.148
Promedio heurística: 0.124
Tiempo promedio: 0.032060582399368286



## Discuta

**¿Qué tan frecuentemente Hill Climbing encuentra una solución?**

En 1000 ejecuciones con estado inicial aleatorio, Hill Climbing encuentra una solución en aproximadamente 15% de los casos (por ejemplo, 14.6% en una corrida típica). Es decir, la mayoría de las veces el algoritmo se estanca en un máximo local y no llega a una configuración con cero conflictos. La baja tasa de éxito se debe a que el espacio de búsqueda tiene muchos máximos locales y la calidad del estado inicial determina en gran medida si se alcanza el óptimo global.

**¿Qué tipo de problemas presenta Hill Climbing en este dominio?**

Hill Climbing sufre de máximos locales: en muchas ejecuciones deja de mejorar porque ningún vecino tiene heurística menor que la actual, aunque la solución global (heurística 0) siga existiendo. Además depende fuertemente del estado inicial; si este cae en la cuenca de atracción de un máximo local, el algoritmo no puede escapar. No hay mecanismo de aleatoriedad ni de reinicio, por lo que una vez atrapado el comportamiento es determinista y limitado.

**¿La variante elegida mejora el desempeño? ¿Por qué?**

Con First-choice Hill Climbing la tasa de éxito es muy similar a la del Hill Climbing estándar (por ejemplo, 14.8% frente a 14.6%). La variante no mejora de forma notable el porcentaje de éxito en este dominio porque ambos comparten la misma limitación: cuando no hay vecinos que mejoren la heurística, ambos se detienen. First-choice introduce aleatoriedad en el orden en que se revisan los vecinos, lo que puede variar ligeramente qué máximo local se alcanza y alarga un poco el episodio en promedio (más pasos hasta estancarse), pero no ofrece una forma sistemática de escapar de máximos locales. Una variante como Random-restart sí podría mejorar el desempeño al reiniciar desde nuevos estados aleatorios tras cada estancamiento.

**¿Cómo afecta el valor de k en Beam Search?**
El valor de k en Beam Search determina cuántos estados se mantienen en cada iteración. Cuando k es pequeño, el algoritmo explora menos alternativas y tiene mayor probabilidad de quedar atrapado en mínimos locales, lo que reduce el porcentaje de éxito y puede aumentar el número de pasos necesarios para encontrar una solución.

A medida que k aumenta, el algoritmo mantiene más candidatos simultáneamente, lo que mejora la exploración del espacio de estados. Esto generalmente incrementa el porcentaje de éxito, reduce el largo promedio de los episodios y produce valores heurísticos finales más cercanos a cero.

Sin embargo, valores mayores de k también implican un mayor costo computacional, ya que se deben generar y evaluar más estados en cada iteración.

En los experimentos realizados, al aumentar k de 2 a 10, el porcentaje de éxito aumentó de 0.489 a 0.876, mientras que el largo promedio del episodio disminuyó de 104.456 a 28.148, lo que confirma que valores mayores de k permiten encontrar soluciones más eficientemente.

**¿Cuál algoritmo resultó más efectivo?**

El algoritmo más efectivo fue Beam Search con k=10, ya que obtuvo el mayor porcentaje de éxito (0.876), el menor largo promedio de episodio (28.148) y el menor valor heurístico final (0.124). Estos resultados indican que el algoritmo no solo encuentra soluciones con mayor frecuencia, sino que también llega a ellas en menos pasos y termina más cerca de una solución incluso cuando no tiene éxito.

En comparación con valores menores de k, mantener más estados en el beam permite explorar mejor el espacio de búsqueda y reduce la probabilidad de quedar atrapado en mínimos locales, lo que mejora el desempeño general del algoritmo.

**¿Qué relación existe entre costo computacional y tasa de éxito?**

En Hill Climbing y First-choice el tiempo por ejecución es muy bajo (del orden de décimas o milésimas de segundo por corrida) porque cada paso solo evalúa vecinos hasta encontrar mejora o agotarlos. First-choice suele evaluar menos vecinos por paso en promedio al tomar el primero que mejore, pero puede dar episodios algo más largos (más pasos) por la aleatoriedad. En estos dos algoritmos no hay un trade-off fuerte entre costo y éxito: gastar más tiempo no aumenta la tasa de éxito, ya que no hay reinicios ni exploración adicional. Para mejorar la tasa de éxito haría falta un mecanismo que incremente el costo (por ejemplo, múltiples reinicios o un beam más ancho en Beam Search) a cambio de explorar más el espacio.

En Beam Search este trade-off se observa claramente. Al aumentar el valor de k, el algoritmo mantiene más estados en cada iteración, lo que incrementa el costo computacional al generar y evaluar más vecinos. Sin embargo, esto también mejora la exploración del espacio de búsqueda, lo que se refleja en un mayor porcentaje de éxito y menos pasos promedio para encontrar una solución.

## Conclusiones

Este laboratorio evidenció que, en el problema de las 8 reinas, la estrategia de exploración del espacio de estados es el factor que más impacta el desempeño. Tanto Hill Climbing como First-choice Hill Climbing mostraron tasas de éxito bajas porque dependen fuertemente del estado inicial y se detienen al llegar a máximos locales sin un mecanismo explícito de escape.

Beam Search, en cambio, obtuvo un desempeño claramente superior al mantener múltiples candidatos en paralelo. Los resultados para k = 2, 5 y 10 muestran una tendencia consistente: al aumentar k, sube la tasa de éxito y disminuyen los pasos promedio para alcanzar solución, además de mejorar la heurística final. Esto confirma que incrementar la diversidad de estados activos reduce la probabilidad de estancamiento.

Sin embargo, esa mejora no es gratuita: un mayor valor de k también incrementa el costo computacional por iteración. Por tanto, la elección del algoritmo y de sus parámetros debe entenderse como un compromiso entre calidad de solución, robustez frente a óptimos locales y tiempo de cómputo disponible.

Como trabajo futuro, sería valioso incorporar variantes con mecanismos de escape más fuertes (por ejemplo, Random-restart Hill Climbing o Simulated Annealing) y extender el análisis a tableros de mayor tamaño (N reinas) para evaluar si las tendencias observadas se mantienen en espacios de búsqueda más complejos.